In [59]:
import torch
import torch.nn as nn
import torch.onnx as onnx
import os
import json
import shutil
from random import randint
from c_exporter.onnx_exporter import export_onnx

In [60]:
fixtures_path = "tests/fixtures"

if os.path.exists(fixtures_path):
    shutil.rmtree(fixtures_path)
os.makedirs(fixtures_path)

In [61]:
def get_output_size() -> int:
    return randint(2, 10) * 5

In [62]:
class SimpleLinearModel(nn.Module):
    def __init__(self, input_size, output_size, layer_num):
        super(SimpleLinearModel, self).__init__()
        self.layers = nn.ModuleList()
        if layer_num == 1:
            self.layers.append(nn.Linear(input_size, output_size))
        else:
            inter_output_size = get_output_size()
            self.layers.append(nn.Linear(input_size, inter_output_size))
            inter_input_size = inter_output_size
            for i in range(layer_num - 2):
                inter_output_size = get_output_size()
                self.layers.append(nn.Linear(inter_input_size, inter_output_size))
                inter_input_size = inter_output_size
            self.layers.append(nn.Linear(inter_input_size, output_size))

    def forward(self, x):
        # Pass input through the linear layer
        output = x
        for layer in self.layers:
            output = layer.forward(output)
        return output

In [63]:
def export_model(name: str, model: nn.Module, input_size: int):
    tmp_model_path = "temporary_model.onnx"
    onnx.export(model, torch.randn(1, input_size), tmp_model_path, export_params=True, opset_version=11)
    dummy_input_data = torch.randn(1, input_size, dtype=torch.float32)
    with torch.no_grad():
        output = model(dummy_input_data)
    model_output = {
        "model": export_onnx(tmp_model_path),
        "test_input": dummy_input_data.tolist()[0],
        "test_output": output.tolist()[0],
    }

    with open(os.path.join(fixtures_path, name), "w") as f:
        json.dump(model_output, f, indent=2)

In [64]:
fixtures = [
    {
        "name": "dense_simple_model.json",
        "model": SimpleLinearModel(10, 5, 1),
        "input_size": 10,
    },
    {
        "name": "dense_long_model.json",
        "model": SimpleLinearModel(10, 5, 20),
        "input_size": 10,
    },
    {
        "name": "dense_large_model.json",
        "model": SimpleLinearModel(100, 100, 5),
        "input_size": 100,
    },
]

In [65]:
for fixture in fixtures:
    export_model(fixture["name"], fixture["model"], fixture["input_size"])